# Athena emulator parity — boto3 control plane

Second notebook of the `athena_emulator` parity series: exercises the boto3 control-plane surface against the live stack — workgroups (seeded `primary`, managed results, `OutputLocation` enforcement), named queries, prepared statements, data catalogs, tags, and engine versions. Each probe records a measured `PASS`/`FAIL`/`GAP`; the session matrix persists to `PARITY.md`. Error probes assert the shaped `ClientError` (`Code` + HTTP status) a real consumer sees.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid
from dataclasses import dataclass

import boto3
from _evidence import persist, record, render
from _helpers import (
    client_error_status,
    delete_run_buckets,
    delete_run_databases,
    delete_workgroup_contents,
    error_code_status,
    object_keys_under,
    query_execution,
    run_workgroup_names,
)
from botocore.exceptions import ClientError

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL

REGION = os.environ["AWS_DEFAULT_REGION"]
ACCOUNT_ID = "123456789012"  # accepted but not enforced by the emulator
RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb02- prefix (nb02_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb02-"
DATABASE_PREFIX = "nb02_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
crud_workgroup = f"{ATHENA_RESOURCE_PREFIX}wg-{RUN_SUFFIX}"
enforced_workgroup = f"{ATHENA_RESOURCE_PREFIX}enf-{RUN_SUFFIX}"
fallback_workgroup = f"{ATHENA_RESOURCE_PREFIX}fb-{RUN_SUFFIX}"
managed_workgroup = f"{ATHENA_RESOURCE_PREFIX}mgr-{RUN_SUFFIX}"
disabled_workgroup = f"{ATHENA_RESOURCE_PREFIX}dis-{RUN_SUFFIX}"
catalog_name = f"{ATHENA_RESOURCE_PREFIX}cat-{RUN_SUFFIX}"
disposable_catalog = f"{ATHENA_RESOURCE_PREFIX}del-{RUN_SUFFIX}"
# Model pattern [a-zA-Z_][a-zA-Z0-9_@:]{1,256} — no dash, no leading digit.
statement_name = f"nb02_stmt_{RUN_SUFFIX}"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})

{'ResponseMetadata': {'RequestId': '8kUmIi0HyrgxFU8solpoYzXPEqYZp3FQLLOZgdEXSFxO2hkYbcGr',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Werkzeug/3.1.3 Python/3.13.9, amazon.com',
   'date': 'Tue, 06 Oct 2026 23:55:17 GMT, Tue, 06 Oct 2026 23:55:17 GMT',
   'x-amzn-requestid': '8kUmIi0HyrgxFU8solpoYzXPEqYZp3FQLLOZgdEXSFxO2hkYbcGr',
   'content-type': 'text/html; charset=utf-8',
   'content-length': '0',
   'access-control-allow-origin': '*',
   'connection': 'close'},
  'RetryAttempts': 0}}

## Helpers

Shared plumbing for the probes below: unwrap responses, poll queries to a terminal state, capture `ClientError` shapes, build ARNs, and sweep every `nb02`-prefixed resource at the end.

In [3]:
@dataclass(frozen=True)
class QueryRun:
    """Terminal outcome of one StartQueryExecution → poll cycle."""

    query_id: str
    state: str
    latency_ms: float


def reported_output_location(query_id: str) -> str | None:
    """The client-facing artifact path GetQueryExecution reports."""
    return (
        query_execution(athena, query_id)
        .get("ResultConfiguration", {})
        .get("OutputLocation")
    )


def run_select_to_terminal(
    workgroup_name: str, output_location: str | None
) -> QueryRun:
    """Start `SELECT 1` on ``workgroup_name`` and poll to a terminal state."""
    request: dict[str, object] = {
        "QueryString": "SELECT 1 AS one",
        "QueryExecutionContext": {"Database": database},
        "WorkGroup": workgroup_name,
    }
    if output_location is not None:
        request["ResultConfiguration"] = {"OutputLocation": output_location}
    started = time.monotonic()
    query_id = athena.start_query_execution(**request)["QueryExecutionId"]
    state = "QUEUED"
    while state in {"QUEUED", "RUNNING"}:
        time.sleep(0.25)
        state = query_execution(athena, query_id)["Status"]["State"]
    return QueryRun(query_id, state, (time.monotonic() - started) * 1000)


def attempted_select(
    workgroup_name: str, output_location: str | None
) -> QueryRun | tuple[str, int]:
    """`SELECT 1` that may be rejected: QueryRun, or (code, status)."""
    try:
        return run_select_to_terminal(workgroup_name, output_location)
    except ClientError as exc:
        return error_code_status(exc)


def workgroup(name: str) -> dict[str, object]:
    """The GetWorkGroup ``WorkGroup`` member for ``name``."""
    return athena.get_work_group(WorkGroup=name)["WorkGroup"]


def named_query(query_id: str) -> dict[str, object]:
    """The GetNamedQuery ``NamedQuery`` member for ``query_id``."""
    return athena.get_named_query(NamedQueryId=query_id)["NamedQuery"]


def prepared_statement(name: str, workgroup_name: str) -> dict[str, object]:
    """The GetPreparedStatement ``PreparedStatement`` member."""
    return athena.get_prepared_statement(
        StatementName=name, WorkGroup=workgroup_name
    )["PreparedStatement"]


def data_catalog(name: str) -> dict[str, object]:
    """The GetDataCatalog ``DataCatalog`` member for ``name``."""
    return athena.get_data_catalog(Name=name)["DataCatalog"]


def resource_arn(resource_type: str, resource_name: str) -> str:
    """``arn:aws:athena:<region>:<account>:<type>/<name>`` — the tag ARN."""
    return (
        f"arn:aws:athena:{REGION}:{ACCOUNT_ID}:{resource_type}/{resource_name}"
    )


def delete_run_catalogs() -> None:
    """Delete every data catalog carrying the run prefix."""
    summaries = athena.list_data_catalogs()["DataCatalogsSummary"]
    for summary in summaries:
        if not summary["CatalogName"].startswith(ATHENA_RESOURCE_PREFIX):
            continue
        athena.delete_data_catalog(Name=summary["CatalogName"])


def cleanup_run_resources() -> None:
    """Remove everything this run (or a failed earlier run) left behind."""
    for name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
        delete_workgroup_contents(athena, name)
        athena.delete_work_group(WorkGroup=name)
    delete_run_catalogs()
    delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
    delete_run_databases(glue, DATABASE_PREFIX)

## Workgroups — `primary` seed and CRUD

`primary` exists out of the box (`ENABLED`, `EnforceWorkGroupConfiguration=False`, engine `AUTO`→`Athena engine version 3`). Created workgroups get the moto-style defaults (`enforce=True`, engine `AUTO`). `GetWorkGroup.Configuration` must carry `EnforceWorkGroupConfiguration`, `ResultConfiguration.OutputLocation`, and `ManagedQueryResultsConfiguration.Enabled` — the exact members `awswrangler._get_workgroup_config` reads.

In [4]:
primary = workgroup("primary")
primary_config = primary["Configuration"]

assert primary["State"] == "ENABLED"
# Hard index, same as wrangler's workgroup resolution — a missing member
# would crash a consumer, so presence is part of the assertion.
assert primary_config["EnforceWorkGroupConfiguration"] is False
assert primary_config["EngineVersion"] == {
    "SelectedEngineVersion": "AUTO",
    "EffectiveEngineVersion": "Athena engine version 3",
}

record(
    "primary workgroup seeded (ENABLED, enforce=False, engine AUTO/3)",
    "PASS",
    f"state={primary['State']} "
    f"enforce={primary_config['EnforceWorkGroupConfiguration']}",
)

Evidence(feature='primary workgroup seeded (ENABLED, enforce=False, engine AUTO/3)', status='PASS', detail='state=ENABLED enforce=False', latency_ms=None)

In [5]:
athena.create_work_group(
    Name=crud_workgroup,
    Description="nb02 parity probe",
    Configuration={"BytesScannedCutoffPerQuery": 10485760},
    Tags=[{"Key": "project", "Value": "nb02"}],
)
created = workgroup(crud_workgroup)
created_config = created["Configuration"]

assert created_config["EnforceWorkGroupConfiguration"] is True
assert (
    created_config["EngineVersion"]["EffectiveEngineVersion"]
    == "Athena engine version 3"
)
assert created_config["BytesScannedCutoffPerQuery"] == 10485760
assert created["Description"] == "nb02 parity probe"

athena.update_work_group(
    WorkGroup=crud_workgroup,
    Description="nb02 updated",
    ConfigurationUpdates={
        "BytesScannedCutoffPerQuery": 20971520,
        "ResultConfigurationUpdates": {
            "OutputLocation": f"s3://{bucket}/wg-crud/"
        },
    },
)
updated = workgroup(crud_workgroup)
updated_config = updated["Configuration"]

assert updated["Description"] == "nb02 updated"
assert updated_config["BytesScannedCutoffPerQuery"] == 20971520
assert (
    updated_config["ResultConfiguration"]["OutputLocation"]
    == f"s3://{bucket}/wg-crud/"
)

listed_names = {
    summary["Name"] for summary in athena.list_work_groups()["WorkGroups"]
}
assert {"primary", crud_workgroup} <= listed_names

record(
    "workgroup create/get/update/list (defaults applied)",
    "PASS",
    f"{crud_workgroup}: enforce=True default, "
    "BytesScannedCutoffPerQuery 10485760→20971520 via ConfigurationUpdates",
)

Evidence(feature='workgroup create/get/update/list (defaults applied)', status='PASS', detail='nb02-wg-960442d4: enforce=True default, BytesScannedCutoffPerQuery 10485760→20971520 via ConfigurationUpdates', latency_ms=None)

In [6]:
duplicate_error = client_error_status(
    lambda: athena.create_work_group(Name=crud_workgroup)
)
missing_error = client_error_status(lambda: workgroup("nb02-no-such-wg"))
primary_delete_error = client_error_status(
    lambda: athena.delete_work_group(WorkGroup="primary")
)

# The service model declares no ResourceNotFoundException for workgroup
# ops; missing resources answer InvalidRequestException 400, never 404.
assert duplicate_error == ("InvalidRequestException", 400)
assert missing_error == ("InvalidRequestException", 400)
assert primary_delete_error == ("InvalidRequestException", 400)

record(
    "workgroup error taxonomy (duplicate/missing/primary-delete → 400)",
    "PASS",
    "InvalidRequestException 400 for all three",
)

Evidence(feature='workgroup error taxonomy (duplicate/missing/primary-delete → 400)', status='PASS', detail='InvalidRequestException 400 for all three', latency_ms=None)

## `OutputLocation` resolution

An **enforced** workgroup's `ResultConfiguration.OutputLocation` overrides whatever the client passes; a **non-enforced** workgroup's is a fallback when the request omits it (wrangler resolves in the same request→workgroup order). `GetQueryExecution.ResultConfiguration.OutputLocation` reports the full artifact path `{prefix}{QueryID}.csv`, so the S3 listing proves where the file actually landed.

In [7]:
athena.create_work_group(
    Name=enforced_workgroup,
    Configuration={
        "ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/wg-enforced/"
        },
        "EnforceWorkGroupConfiguration": True,
    },
)
enforced_run = run_select_to_terminal(
    enforced_workgroup, f"s3://{bucket}/client-prefix/"
)
assert enforced_run.state == "SUCCEEDED"

enforced_location = reported_output_location(enforced_run.query_id)
assert (
    enforced_location
    == f"s3://{bucket}/wg-enforced/{enforced_run.query_id}.csv"
)
assert f"wg-enforced/{enforced_run.query_id}.csv" in object_keys_under(
    s3, bucket, "wg-enforced/"
)
assert object_keys_under(s3, bucket, "client-prefix/") == []

record(
    "enforced workgroup overrides client OutputLocation",
    "PASS",
    f"client passed s3://{bucket}/client-prefix/; "
    f"reported+written={enforced_location}",
    latency_ms=enforced_run.latency_ms,
)

Evidence(feature='enforced workgroup overrides client OutputLocation', status='PASS', detail='client passed s3://nb02-081374809181/client-prefix/; reported+written=s3://nb02-081374809181/wg-enforced/47e4d64d-8fe7-4abf-935d-4b66cecb4781.csv', latency_ms=274.6536559998276)

In [8]:
athena.create_work_group(
    Name=fallback_workgroup,
    Configuration={
        "ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/wg-fallback/"
        },
        "EnforceWorkGroupConfiguration": False,
    },
)
fallback_run = run_select_to_terminal(fallback_workgroup, None)
assert fallback_run.state == "SUCCEEDED"

fallback_location = reported_output_location(fallback_run.query_id)
assert (
    fallback_location
    == f"s3://{bucket}/wg-fallback/{fallback_run.query_id}.csv"
)
assert f"wg-fallback/{fallback_run.query_id}.csv" in object_keys_under(
    s3, bucket, "wg-fallback/"
)

record(
    "non-enforced workgroup OutputLocation fallback",
    "PASS",
    "request omitted ResultConfiguration; "
    f"reported+written={fallback_location}",
    latency_ms=fallback_run.latency_ms,
)

Evidence(feature='non-enforced workgroup OutputLocation fallback', status='PASS', detail='request omitted ResultConfiguration; reported+written=s3://nb02-081374809181/wg-fallback/6373e572-fab4-4a98-bd3c-49b3de653d83.csv', latency_ms=265.44249600010517)

## Managed-results workgroup

`ManagedQueryResultsConfiguration.Enabled=True` makes Athena own the results: a client-passed `OutputLocation` is ignored, `GetQueryExecution` reports a `ResultConfiguration` **without** `OutputLocation`, no artifacts are written to S3, and rows still come back inline via `GetQueryResults`.

In [9]:
athena.create_work_group(
    Name=managed_workgroup,
    Configuration={"ManagedQueryResultsConfiguration": {"Enabled": True}},
)
managed_run = run_select_to_terminal(
    managed_workgroup, f"s3://{bucket}/ignored/"
)
assert managed_run.state == "SUCCEEDED"

managed_config = query_execution(athena, managed_run.query_id).get(
    "ResultConfiguration", {}
)
assert "OutputLocation" not in managed_config

managed_rows = athena.get_query_results(QueryExecutionId=managed_run.query_id)[
    "ResultSet"
]["Rows"]
assert managed_rows[1]["Data"][0]["VarCharValue"] == "1"
assert object_keys_under(s3, bucket, "ignored/") == []

record(
    "managed-results workgroup (no OutputLocation, inline rows, no S3 writes)",
    "PASS",
    f"ResultConfiguration={managed_config}; "
    f"rows={len(managed_rows) - 1}; ignored/ empty",
    latency_ms=managed_run.latency_ms,
)

Evidence(feature='managed-results workgroup (no OutputLocation, inline rows, no S3 writes)', status='PASS', detail='ResultConfiguration={}; rows=1; ignored/ empty', latency_ms=262.83768999928725)

## Disabled workgroup probe

Real Athena rejects `StartQueryExecution` on a `DISABLED` workgroup. Whatever the emulator does is measured — a rejection is `PASS`, an executed query is a `GAP` for triage.

In [10]:
feature = "disabled workgroup rejects StartQueryExecution"

athena.create_work_group(Name=disabled_workgroup)
athena.update_work_group(WorkGroup=disabled_workgroup, State="DISABLED")
assert workgroup(disabled_workgroup)["State"] == "DISABLED"

disabled_outcome = attempted_select(
    disabled_workgroup, f"s3://{bucket}/disabled/"
)
if isinstance(disabled_outcome, QueryRun):
    record(
        feature,
        "GAP",
        f"{disabled_outcome.query_id} reached {disabled_outcome.state} "
        "on a DISABLED workgroup; real AWS rejects at submit",
    )
if not isinstance(disabled_outcome, QueryRun):
    error_code, http_status = disabled_outcome
    record(feature, "PASS", f"{error_code} ({http_status})")

## Named queries

Workgroup-scoped CRUD plus `BatchGetNamedQuery` (missing IDs land in `UnprocessedNamedQueryIds`) and `ListNamedQueries` `MaxResults`/`NextToken` pagination. Missing queries answer `InvalidRequestException` 400 — the model declares no `ResourceNotFoundException` here either.

In [11]:
first_query_id = athena.create_named_query(
    Name=f"nb02-first-{RUN_SUFFIX}",
    Database=database,
    QueryString="SELECT 1",
    WorkGroup=crud_workgroup,
    Description="first",
)["NamedQueryId"]
second_query_id = athena.create_named_query(
    Name=f"nb02-second-{RUN_SUFFIX}",
    Database=database,
    QueryString="SELECT 2",
    WorkGroup=crud_workgroup,
)["NamedQueryId"]

fetched_query = named_query(first_query_id)
assert fetched_query["Name"] == f"nb02-first-{RUN_SUFFIX}"
assert fetched_query["Database"] == database
assert fetched_query["QueryString"] == "SELECT 1"
assert fetched_query["WorkGroup"] == crud_workgroup

first_page = athena.list_named_queries(WorkGroup=crud_workgroup, MaxResults=1)
second_page = athena.list_named_queries(
    WorkGroup=crud_workgroup,
    MaxResults=1,
    NextToken=first_page["NextToken"],
)
assert first_page["NamedQueryIds"] + second_page["NamedQueryIds"] == [
    first_query_id,
    second_query_id,
]
assert "NextToken" not in second_page

record(
    "named queries create/get/list (MaxResults+NextToken pagination)",
    "PASS",
    f"2 queries in {crud_workgroup}, "
    f"first-page token={first_page['NextToken']!r}",
)

Evidence(feature='named queries create/get/list (MaxResults+NextToken pagination)', status='PASS', detail="2 queries in nb02-wg-960442d4, first-page token='1'", latency_ms=None)

In [12]:
unknown_query_id = "00000000-0000-0000-0000-000000000000"
named_batch = athena.batch_get_named_query(
    NamedQueryIds=[first_query_id, unknown_query_id]
)
assert [q["NamedQueryId"] for q in named_batch["NamedQueries"]] == [
    first_query_id
]
assert named_batch["UnprocessedNamedQueryIds"] == [
    {"NamedQueryId": unknown_query_id}
]

athena.delete_named_query(NamedQueryId=second_query_id)
deleted_lookup = client_error_status(lambda: named_query(second_query_id))
assert deleted_lookup == ("InvalidRequestException", 400)

record(
    "named queries batch_get unprocessed IDs + delete→400",
    "PASS",
    f"unprocessed={[unknown_query_id]}; deleted get → InvalidRequestException",
)

Evidence(feature='named queries batch_get unprocessed IDs + delete→400', status='PASS', detail="unprocessed=['00000000-0000-0000-0000-000000000000']; deleted get → InvalidRequestException", latency_ms=None)

## Prepared statements

Workgroup-scoped CRUD plus `BatchGetPreparedStatement`. Unlike named queries, missing statements answer `ResourceNotFoundException` 404 — the contract awswrangler's prepared-statement path depends on.

In [13]:
athena.create_prepared_statement(
    StatementName=statement_name,
    WorkGroup=crud_workgroup,
    QueryStatement="SELECT * FROM nb02_tbl WHERE x = ?",
    Description="nb02 statement",
)
statement = prepared_statement(statement_name, crud_workgroup)
assert statement["QueryStatement"] == "SELECT * FROM nb02_tbl WHERE x = ?"
assert statement["WorkGroupName"] == crud_workgroup
assert "LastModifiedTime" in statement

athena.update_prepared_statement(
    StatementName=statement_name,
    WorkGroup=crud_workgroup,
    QueryStatement="SELECT * FROM nb02_tbl WHERE x = ? AND y = ?",
    Description="nb02 statement v2",
)
updated_statement = prepared_statement(statement_name, crud_workgroup)
assert updated_statement["QueryStatement"].endswith("AND y = ?")
assert updated_statement["Description"] == "nb02 statement v2"

listed_statements = athena.list_prepared_statements(WorkGroup=crud_workgroup)[
    "PreparedStatements"
]
assert [s["StatementName"] for s in listed_statements] == [statement_name]

statement_batch = athena.batch_get_prepared_statement(
    PreparedStatementNames=[statement_name, "nb02_missing"],
    WorkGroup=crud_workgroup,
)
assert [s["StatementName"] for s in statement_batch["PreparedStatements"]] == [
    statement_name
]
assert statement_batch["UnprocessedPreparedStatementNames"] == [
    {"StatementName": "nb02_missing"}
]

record(
    "prepared statements create/get/update/list/batch_get",
    "PASS",
    f"{statement_name} in {crud_workgroup}; unprocessed names surfaced",
)

Evidence(feature='prepared statements create/get/update/list/batch_get', status='PASS', detail='nb02_stmt_960442d4 in nb02-wg-960442d4; unprocessed names surfaced', latency_ms=None)

In [14]:
missing_get = client_error_status(
    lambda: prepared_statement("nb02_missing", crud_workgroup)
)
missing_update = client_error_status(
    lambda: athena.update_prepared_statement(
        StatementName="nb02_missing",
        WorkGroup=crud_workgroup,
        QueryStatement="SELECT 0",
    )
)
assert missing_get == ("ResourceNotFoundException", 404)
assert missing_update == ("ResourceNotFoundException", 404)

athena.delete_prepared_statement(
    StatementName=statement_name, WorkGroup=crud_workgroup
)
deleted_get = client_error_status(
    lambda: prepared_statement(statement_name, crud_workgroup)
)
assert deleted_get == ("ResourceNotFoundException", 404)

record(
    "prepared statements missing/deleted → ResourceNotFoundException 404",
    "PASS",
    "get/update/get-after-delete all 404",
)

Evidence(feature='prepared statements missing/deleted → ResourceNotFoundException 404', status='PASS', detail='get/update/get-after-delete all 404', latency_ms=None)

## Data catalogs

`AwsDataCatalog` (GLUE) is pre-registered like real AWS. LAMBDA catalogs get their `Parameters.function` normalized into `catalog` + `metadata-function` + `record-function` — the shape the CLI `get-data-catalog` example shows. `FEDERATED` is rejected loudly (400); missing catalogs answer 400, and `DeleteDataCatalog` echoes the deleted catalog.

In [15]:
seeded_catalog = data_catalog("AwsDataCatalog")
assert seeded_catalog["Type"] == "GLUE"
listed_catalog_names = {
    summary["CatalogName"]
    for summary in athena.list_data_catalogs()["DataCatalogsSummary"]
}
assert "AwsDataCatalog" in listed_catalog_names

lambda_function_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:nb02"
athena.create_data_catalog(
    Name=catalog_name,
    Type="LAMBDA",
    Description="nb02 catalog",
    Parameters={"function": lambda_function_arn},
)
fetched_catalog = data_catalog(catalog_name)
assert fetched_catalog["Parameters"]["catalog"] == catalog_name
assert fetched_catalog["Parameters"]["metadata-function"] == (
    lambda_function_arn
)
assert fetched_catalog["Parameters"]["record-function"] == (
    lambda_function_arn
)

athena.update_data_catalog(
    Name=catalog_name,
    Type="LAMBDA",
    Description="nb02 catalog updated",
    Parameters={"function": lambda_function_arn},
)
assert data_catalog(catalog_name)["Description"] == "nb02 catalog updated"

athena.create_data_catalog(
    Name=disposable_catalog,
    Type="HIVE",
    Parameters={"metadata-function": lambda_function_arn},
)
deleted_catalog = athena.delete_data_catalog(Name=disposable_catalog)[
    "DataCatalog"
]
assert deleted_catalog["Name"] == disposable_catalog
assert client_error_status(lambda: data_catalog(disposable_catalog)) == (
    "InvalidRequestException",
    400,
)

federated_error = client_error_status(
    lambda: athena.create_data_catalog(
        Name=f"{catalog_name}fed", Type="FEDERATED"
    )
)
assert federated_error == ("InvalidRequestException", 400)

record(
    "data catalogs CRUD + AwsDataCatalog seed + LAMBDA normalization",
    "PASS",
    "AwsDataCatalog=GLUE; function→catalog/metadata-function/"
    "record-function; FEDERATED→400; missing→400",
)

Evidence(feature='data catalogs CRUD + AwsDataCatalog seed + LAMBDA normalization', status='PASS', detail='AwsDataCatalog=GLUE; function→catalog/metadata-function/record-function; FEDERATED→400; missing→400', latency_ms=None)

## Tags

Resources are addressed by Athena ARN (`arn:aws:athena:<region>:<account>:<type>/<name>`); `workgroup` and `datacatalog` are the taggable types. Create-time tags must already be on the workgroup; `TagResource`/`UntagResource`/`ListTagsForResource` mutate them. Unknown resources → 404 `ResourceNotFoundException`; malformed ARNs and unmodelled types → 400.

In [16]:
workgroup_arn = resource_arn("workgroup", crud_workgroup)
catalog_arn = resource_arn("datacatalog", catalog_name)

initial_tags = athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
assert {"Key": "project", "Value": "nb02"} in initial_tags

athena.tag_resource(
    ResourceARN=workgroup_arn, Tags=[{"Key": "owner", "Value": "nb02"}]
)
tagged_keys = {
    tag["Key"]
    for tag in athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
}
assert tagged_keys == {"project", "owner"}

athena.untag_resource(ResourceARN=workgroup_arn, TagKeys=["project"])
remaining_keys = {
    tag["Key"]
    for tag in athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
}
assert remaining_keys == {"owner"}

athena.tag_resource(
    ResourceARN=catalog_arn, Tags=[{"Key": "kind", "Value": "parity"}]
)
assert athena.list_tags_for_resource(ResourceARN=catalog_arn)["Tags"] == [
    {"Key": "kind", "Value": "parity"}
]

record(
    "tags tag/untag/list on workgroup + datacatalog ARNs",
    "PASS",
    "create-time tag present; +owner/-project on workgroup; "
    "kind=parity on datacatalog",
)

Evidence(feature='tags tag/untag/list on workgroup + datacatalog ARNs', status='PASS', detail='create-time tag present; +owner/-project on workgroup; kind=parity on datacatalog', latency_ms=None)

In [17]:
unknown_resource_error = client_error_status(
    lambda: athena.list_tags_for_resource(
        ResourceARN=resource_arn("workgroup", "nb02-no-such")
    )
)
malformed_arn_error = client_error_status(
    lambda: athena.tag_resource(
        ResourceARN="not-an-arn",
        Tags=[{"Key": "k", "Value": "v"}],
    )
)
untaggable_type_error = client_error_status(
    lambda: athena.list_tags_for_resource(
        ResourceARN=resource_arn("preparedstatement", "x")
    )
)

assert unknown_resource_error == ("ResourceNotFoundException", 404)
assert malformed_arn_error == ("InvalidRequestException", 400)
assert untaggable_type_error == ("InvalidRequestException", 400)

record(
    "tag error taxonomy (unknown → 404, malformed/non-taggable → 400)",
    "PASS",
    "RNFE 404 for unknown workgroup ARN; "
    "InvalidRequestException for bad ARN and preparedstatement type",
)

Evidence(feature='tag error taxonomy (unknown → 404, malformed/non-taggable → 400)', status='PASS', detail='RNFE 404 for unknown workgroup ARN; InvalidRequestException for bad ARN and preparedstatement type', latency_ms=None)

## `ListWorkGroups` pagination probe

The service model declares `MaxResults`/`NextToken` on `ListWorkGroups`. Measured: whether the emulator honors a `MaxResults=1` page size or returns the full list.

In [18]:
feature = "list_work_groups honors MaxResults/NextToken"

paged_listing = athena.list_work_groups(MaxResults=1)
page_size_respected = (
    "NextToken" in paged_listing and len(paged_listing["WorkGroups"]) == 1
)
if page_size_respected:
    record(
        feature,
        "PASS",
        f"1 workgroup + NextToken={paged_listing['NextToken']!r}",
    )
if not page_size_respected:
    record(
        feature,
        "GAP",
        f"MaxResults=1 returned {len(paged_listing['WorkGroups'])} "
        f"workgroups, NextToken={'NextToken' in paged_listing}",
    )

## Engine versions

The emulator exposes one engine — the Trino-backed `Athena engine version 3` — matching the workgroup default.

In [19]:
engine_versions = athena.list_engine_versions()["EngineVersions"]
assert engine_versions == [
    {
        "SelectedEngineVersion": "AUTO",
        "EffectiveEngineVersion": "Athena engine version 3",
    }
]

record(
    "list_engine_versions (single AUTO/engine-3 entry)",
    "PASS",
    f"{len(engine_versions)} version(s): "
    f"{engine_versions[0]['EffectiveEngineVersion']}",
)

Evidence(feature='list_engine_versions (single AUTO/engine-3 entry)', status='PASS', detail='1 version(s): Athena engine version 3', latency_ms=None)

## Session matrix

Persisted to `parity/02_boto3_control_plane.md`; `PARITY.md` at the project root is regenerated from all fragments.

In [20]:
print(render())
persist("02_boto3_control_plane")

| feature | status | detail | latency_ms |
|---|---|---|---|
| primary workgroup seeded (ENABLED, enforce=False, engine AUTO/3) | PASS | state=ENABLED enforce=False |  |
| workgroup create/get/update/list (defaults applied) | PASS | nb02-wg-960442d4: enforce=True default, BytesScannedCutoffPerQuery 10485760→20971520 via ConfigurationUpdates |  |
| workgroup error taxonomy (duplicate/missing/primary-delete → 400) | PASS | InvalidRequestException 400 for all three |  |
| enforced workgroup overrides client OutputLocation | PASS | client passed s3://nb02-081374809181/client-prefix/; reported+written=s3://nb02-081374809181/wg-enforced/47e4d64d-8fe7-4abf-935d-4b66cecb4781.csv | 274.654 |
| non-enforced workgroup OutputLocation fallback | PASS | request omitted ResultConfiguration; reported+written=s3://nb02-081374809181/wg-fallback/6373e572-fab4-4a98-bd3c-49b3de653d83.csv | 265.442 |
| managed-results workgroup (no OutputLocation, inline rows, no S3 writes) | PASS | ResultConfiguration={}; 

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/02_boto3_control_plane.md')

In [21]:
cleanup_run_resources()
print(
    "cleaned: "
    f"workgroups={run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX)} "
    f"catalogs+buckets+databases with the nb02 prefix removed"
)

cleaned: workgroups=[] catalogs+buckets+databases with the nb02 prefix removed
